# CDFI DiD Estimation Pipeline v2.0

Callaway & Sant'Anna (2021) Difference-in-Differences with Neural Network Nuisance Estimation

**Key improvements in v2.0:**
- Wide unit-level data structure
- Base-period grouped covariate projections (~19 instead of ~342)
- Single forward pass produces all (g,t) predictions
- On-the-fly outcome differencing

---

## Cell 1: Setup

In [ ]:
import sys
import os

# Set project root
project_root = "/path/to/your/project"  # <-- CHANGE THIS
os.chdir(project_root)
sys.path.insert(0, os.path.join(project_root, "code/python"))

# Import v2.0 modules
from modules import (
    # Config
    create_config, set_seed, print_config, get_device,
    # Data
    load_panel_data, create_unit_data, create_gt_info,
    # Covariates
    create_covariate_info,
    # Cross-fitting
    run_cross_fitting, validate_cross_fitting,
    # ATT
    compute_all_att, print_att_summary,
    # Inference
    add_bootstrap_inference, test_parallel_trends, compute_simple_att,
    # Aggregation
    aggregate_all, print_aggregation_summary,
    # Visualization
    plot_event_study, save_event_study,
    # Utils
    Timer
)

import torch
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"MPS: {torch.backends.mps.is_available()}")
print("\nSetup complete.")

## Cell 2: Configuration

In [ ]:
config = create_config(
    # Outcome
    outcome="sfr_pc",
    
    # Data structure
    id_var="id",
    time_var="time",
    group_var="group",
    cluster_var="cluster_county",
    never_treated_code=0,
    
    # Analysis period
    analysis_start=1996,
    analysis_end=2014,
    
    # Architecture
    architecture={
        'input_projection_dim': 128,
        'shared_layers': [256, 128],
        'outcome_head_layers': [64],
        'propensity_head_layers': [64],
        'activation': 'relu',
        'dropout': 0.2,
        'layer_norm': True
    },
    
    # Training
    training={
        'epochs': 100,
        'batch_size': 512,
        'validation_split': 0.2,
        'early_stopping_patience': 15,
        'gradient_clip_norm': 1.0
    },
    
    # Optimizer
    optimizer={'name': 'adamw', 'lr': 0.001, 'weight_decay': 0.01},
    
    # Scheduler
    scheduler={'name': 'cosine', 'T_max': 100, 'eta_min': 1e-6},
    
    # Cross-fitting
    cross_fitting={'n_folds': 2, 'stratify_by': 'cluster_county', 'seed': 42},
    
    # Propensity
    propensity={'min_ps': 0.001, 'max_ps': 0.999},
    
    # Inference
    inference={
        'n_bootstrap': 1000,
        'alpha': 0.05,
        'uniform_bands': True,
        'multiplier_dist': 'normal',
        'seed': 123
    },
    
    # Event study
    event_study={
        'pre_periods': 10,
        'post_periods': 10,
        'reference_period': -1,
        'weight_by_group_size': True
    },
    
    # Monitoring
    monitoring={'verbose': True, 'print_every': 10},
    
    # Computational
    device='auto',
    seed=42
)

print_config(config)
set_seed(config.seed)
device = get_device(config)
print(f"Device: {device}")

## Cell 3: Load Data and Create Unit-Level Structure

In [ ]:
data_path = os.path.join(project_root, "data/analysis/final_analysis_dataset.csv")

# Sample size for testing (None for full data)
sample_n = None  # e.g., 10000 for testing

timer = Timer()

# Load panel data
panel_df = load_panel_data(data_path, config, sample_n=sample_n)

# Create wide unit-level structure
unit_data = create_unit_data(panel_df, config)

# Create (g,t) pair info
gt_info = create_gt_info(unit_data, config)

# Create covariate masks by base period
covariate_info = create_covariate_info(unit_data, gt_info, config)

print(f"\nData preparation complete in {timer.elapsed():.1f}s")
print(f"\nSummary:")
print(f"  Units: {unit_data.n_units:,}")
print(f"  Times: {unit_data.n_times}")
print(f"  Treatment groups: {len(unit_data.treatment_groups)}")
print(f"  (g,t) pairs: {gt_info.n_gt}")
print(f"  Unique base periods: {len(gt_info.unique_base_periods)}")
print(f"  Covariates: {covariate_info.n_covariates}")

## Cell 4: Cross-Fitting

**This is the computationally intensive step.** Each fold trains the model and computes out-of-fold predictions for all (g,t) pairs in a single forward pass.

In [ ]:
print("=" * 60)
print("CROSS-FITTING")
print("=" * 60)

timer = Timer()

cf_results = run_cross_fitting(
    unit_data,
    gt_info,
    covariate_info,
    config
)

training_time = timer.elapsed()
print(f"\nCross-fitting complete in {training_time:.1f}s ({training_time/60:.1f}m)")

# Validate
validate_cross_fitting(cf_results)

## Cell 5: ATT Estimation

In [ ]:
print("=" * 60)
print("ATT ESTIMATION")
print("=" * 60)

timer = Timer()

att_results = compute_all_att(cf_results, config)
print_att_summary(att_results)

print(f"\nATT estimation complete in {timer.elapsed():.1f}s")

## Cell 6: Inference (Bootstrap)

In [ ]:
print("=" * 60)
print("INFERENCE")
print("=" * 60)

timer = Timer()

# Add bootstrap inference
att_results = add_bootstrap_inference(att_results, config)

# Test parallel trends
pt_test = test_parallel_trends(att_results, config)

# Simple ATT
simple_att = compute_simple_att(att_results, config)

print(f"\nInference complete in {timer.elapsed():.1f}s")

print("\n--- Parallel Trends Test ---")
print(f"Mean pre-treatment ATT: {pt_test['mean_att_pre']:.4f} (SE: {pt_test['se_mean_pre']:.4f})")
print(f"p-value: {pt_test['p_value']:.4f}")
print(f"Reject at 5%: {'YES' if pt_test['reject'] else 'NO'}")

print("\n--- Simple ATT ---")
print(f"ATT: {simple_att['att']:.4f} (SE: {simple_att['se']:.4f})")
print(f"95% CI: [{simple_att['ci_lower']:.4f}, {simple_att['ci_upper']:.4f}]")
print(f"p-value: {simple_att['p_value']:.4f}")

## Cell 7: Aggregation

In [ ]:
print("=" * 60)
print("AGGREGATION")
print("=" * 60)

agg_results = aggregate_all(att_results, config)
print_aggregation_summary(agg_results)

# Display event study table
print("\n--- Event Study Estimates ---")
es_df = agg_results['event_study']['event_study']
display_cols = ['event_time', 'att', 'se', 'ci_lower', 'ci_upper']
if 'uniform_lower' in es_df.columns:
    display_cols += ['uniform_lower', 'uniform_upper']
print(es_df[display_cols].to_string(index=False))

## Cell 8: Visualization

In [ ]:
print("=" * 60)
print("VISUALIZATION")
print("=" * 60)

output_dir = os.path.join(project_root, "outputs/figures")
os.makedirs(output_dir, exist_ok=True)

# Event study plot
fig = plot_event_study(
    agg_results['event_study'],
    title="Effect of CDFI Lending on Startup Formation Rate",
    subtitle="Callaway & Sant'Anna (2021) DiD with Neural Network Nuisance Estimation",
    show_uniform_bands=True,
    show_pointwise_ci=True
)
plt.show()

# Save
save_event_study(fig, "event_study", output_dir)
print(f"\nFigures saved to: {output_dir}")

## Cell 9: Save Results

In [ ]:
import pickle

results = {
    'config': config,
    'unit_data': unit_data,
    'gt_info': gt_info,
    'covariate_info': covariate_info,
    'cf_results': cf_results,
    'att_results': att_results,
    'agg_results': agg_results,
    'parallel_trends_test': pt_test,
    'simple_att': simple_att,
    'training_time': training_time
}

output_path = os.path.join(project_root, "outputs/estimation_results.pkl")
with open(output_path, 'wb') as f:
    pickle.dump(results, f)

print(f"Results saved to: {output_path}")

## Cell 10: Summary

In [ ]:
print("\n" + "=" * 60)
print("ESTIMATION COMPLETE")
print("=" * 60)

print("\nKEY RESULTS:\n")

print("1. Simple ATT (weighted average post-treatment):")
print(f"   ATT = {simple_att['att']:.4f}, SE = {simple_att['se']:.4f}")
print(f"   95% CI: [{simple_att['ci_lower']:.4f}, {simple_att['ci_upper']:.4f}]")
print(f"   p-value: {simple_att['p_value']:.4f}\n")

print("2. Parallel Trends:")
print(f"   Pre-treatment ATT = {pt_test['mean_att_pre']:.4f} (should be ~0)")
print(f"   p-value = {pt_test['p_value']:.4f} (want > 0.05)\n")

print("3. Event Study:")
es = agg_results['event_study']['event_study']
print(f"   Event times: {es['event_time'].min()} to {es['event_time'].max()}")
print(f"   Pre-treatment mean: {es[es['event_time'] < 0]['att'].mean():.4f}")
print(f"   Post-treatment mean: {es[es['event_time'] >= 0]['att'].mean():.4f}")

print(f"\nTraining time: {training_time/60:.1f} minutes")
print(f"Figures: {output_dir}")
print(f"Results: {output_path}")